# تجهيز بيانات الأداء (Performers) — تحميل الصور + فحص الوجوه

النوتبوك ده بيعمل خطوتين:
1. **تحميل صور الأشخاص** من ملف الـ JSON وتنظيمها في مجلدات + تقرير تحميل.
2. **فحص وجود وجه** في صور أصحاب الصورة الواحدة باستخدام YOLOv11l-face + تقرير فحص.

> ⚠️ لازم تفعّل **GPU** من إعدادات النوتبوك (Settings → Accelerator) وتتأكد إن
> **Internet** مفعّل (Settings → Internet) عشان الـ clone وتحميل الموديل يشتغلوا.


## 0) رابط الـ GitHub Repo

حط رابط الريبو بتاعك هنا (لسه هتعمله). الملفات المتوقعة في **جذر الريبو**:
`01_download_images.py`, `02_detect_faces.py`, `requirements.txt`.


In [ ]:
# 🔗 حط رابط الريبو هنا قبل التشغيل، مثال:
# REPO_URL = "https://github.com/username/repo-name.git"
REPO_URL = ""

assert REPO_URL, "❌ لازم تحط رابط الـ GitHub repo في REPO_URL الأول قبل ما تكمل!"

# 🔑 التوكن بتاع GitHub (Personal Access Token) — لازم يكون معاه صلاحية
# "repo" (أو "contents: write" لو fine-grained) عشان السكربت يقدر يعمل
# push للتقارير في آخر النوتبوك. سيبه فاضي لو مش عايز رفع تلقائي
# (هيتعمل تخطي للخطوة دي بس من غير ما يوقف باقي النوتبوك).
#
# ⚠️ الأفضل أمنياً: متكتبش التوكن هنا كنص صريح. استخدم Kaggle Secrets
# بدل كده (Add-ons → Secrets في الـ Notebook)، وهيتقرا تلقائياً هنا لو
# مسجّل باسم GITHUB_TOKEN. لو مش متسجل هيرجع للمتغير التحتي.
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    _secret = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if _secret:
        GITHUB_TOKEN = _secret
        print("تم قراءة GITHUB_TOKEN من Kaggle Secrets.")
except Exception:
    pass  # مفيش Kaggle Secrets متسجل بالاسم ده، أو مش متاح — هنستخدم المتغير فوق


## 1) فحص الـ GPU المتاح


In [ ]:
!nvidia-smi


## 2) استنساخ (clone) المشروع من GitHub


In [ ]:
import os, shutil

PROJECT_DIR = "/kaggle/working/project"

if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

!git clone "{REPO_URL}" "{PROJECT_DIR}"


In [ ]:
%cd {PROJECT_DIR}
!ls -la


## 3) تثبيت المتطلبات


In [ ]:
!pip install -q -r requirements.txt


## 4) المرحلة 1: تحميل الصور

غيّر `JSON_SOURCE` لو حابب (ده رابط ملف الـ performers على Hugging Face).


In [ ]:
JSON_SOURCE = "https://huggingface.co/datasets/abdelwahabnabil500/datafile/resolve/main/stashdb_performers_full.json"
DOWNLOAD_DIR = "/kaggle/working/downloads"
REPORT_DIR = "/kaggle/working/reports"


In [ ]:
!python 01_download_images.py \
    --json-source "{JSON_SOURCE}" \
    --output-dir "{DOWNLOAD_DIR}" \
    --report-dir "{REPORT_DIR}" \
    --workers 16


### عرض تقرير التحميل


In [ ]:
with open(f"{REPORT_DIR}/download_report.md", encoding="utf-8") as f:
    print(f.read())


## 5) المرحلة 2: فحص وجود الوجه (YOLOv11l-face على GPU)

بيتحمّل وزن الموديل تلقائياً أول مرة (48.8MB) من GitHub، وبيفحص فقط
الأشخاص أصحاب الصورة الواحدة اللي نجح تحميلهم في المرحلة السابقة.


In [ ]:
!python 02_detect_faces.py \
    --download-report "{REPORT_DIR}/download_report.json" \
    --report-dir "{REPORT_DIR}" \
    --device 0


### عرض تقرير فحص الوجوه


In [ ]:
with open(f"{REPORT_DIR}/face_detection_report.md", encoding="utf-8") as f:
    print(f.read())


## 6) ملخص سريع


In [ ]:
import json

with open(f"{REPORT_DIR}/download_report.json", encoding="utf-8") as f:
    dl = json.load(f)
with open(f"{REPORT_DIR}/face_detection_report.json", encoding="utf-8") as f:
    fd = json.load(f)

s = dl["single_image_performers"]
print("إجمالي الأشخاص:", dl["total_performers"])
print("أصحاب الصورة الواحدة:", s["total"])
print("  - نجح تحميلها:", s["downloaded_success"])
print("  - فشل تحميلها:", s["downloaded_failed"])
print("من ضمن اللي اتحمّلت -> فيها وجه:", fd["face_detected"])
print("من ضمن اللي اتحمّلت -> من غير وجه:", fd["face_not_detected"])


## 7) رفع التقارير النهائية على GitHub

بعد ما تقارير `download_report.*` و `face_detection_report.*` تخلص، الخلية اللي جاية بتعمل commit لمجلد `reports/` وتعمل push للريبو اللي حددته في `REPO_URL`.

- لو `GITHUB_TOKEN` فاضي: هيتم تخطي الرفع مع تنبيه، وباقي النوتبوك بيفضل شغال عادي.
- لو مفيش تغييرات جديدة في `reports/` (نفس محتوى آخر commit)، مش هيعمل commit فاضي.
- التوكن بيتبعت لـ GitHub عن طريق `http.extraheader` وقت الـ push بس (مش بيتخزن في أي ملف ومش بيظهر في اللوج).


In [ ]:
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path


def _run(cmd, cwd=None, redact=None):
    """يشغّل أمر ويطبع كل حاجة إلا القيم السرية (زي التوكن)."""
    redact = redact or []
    printable = " ".join(cmd)
    for secret in redact:
        if secret:
            printable = printable.replace(secret, "***")
    print("+", printable)
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if result.stdout.strip():
        print(result.stdout)
    if result.returncode != 0:
        stderr = result.stderr
        for secret in redact:
            if secret:
                stderr = stderr.replace(secret, "***")
        print(stderr)
        raise RuntimeError(f"فشل الأمر: {printable}")
    return result


if not GITHUB_TOKEN:
    print("⚠️ GITHUB_TOKEN فاضي — هيتم تخطي رفع التقارير على GitHub.")
    print("لو عايز الرفع التلقائي، حط التوكن في GITHUB_TOKEN فوق (أو "
          "سجّله في Kaggle Secrets باسم GITHUB_TOKEN) وشغّل من الأول.")
else:
    # التقارير النهائية موجودة في REPORT_DIR (/kaggle/working/reports)،
    # ومشروع الـ git المستنسخ في PROJECT_DIR — نتأكد إنهم نفس المجلد
    # أو ننسخ التقارير جوه مجلد المشروع عشان يبقوا جزء من الريبو.
    reports_src = Path(REPORT_DIR).resolve()
    reports_dst = (Path(PROJECT_DIR) / "reports").resolve()
    if reports_src != reports_dst:
        import shutil
        if reports_dst.exists():
            shutil.rmtree(reports_dst)
        shutil.copytree(reports_src, reports_dst)
        print(f"تم نسخ التقارير من {reports_src} إلى {reports_dst}")

    _run(["git", "config", "user.email", "kaggle-bot@users.noreply.github.com"], cwd=PROJECT_DIR)
    _run(["git", "config", "user.name", "Kaggle Notebook Bot"], cwd=PROJECT_DIR)
    _run(["git", "add", "reports"], cwd=PROJECT_DIR)

    diff_check = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=PROJECT_DIR)
    if diff_check.returncode == 0:
        print("مفيش تغييرات جديدة في reports/ — هيتم تخطي الـ commit والـ push.")
    else:
        commit_msg = f"تحديث التقارير من Kaggle - {datetime.now(timezone.utc).isoformat()}"
        _run(["git", "commit", "-m", commit_msg], cwd=PROJECT_DIR)

        # بعث التوكن عن طريق extraheader وقت الـ push بس (زي ما بتعمل
        # GitHub Actions نفسها) — التوكن مش بيتخزن في remote URL ولا في
        # أي ملف على القرص.
        auth_header = f"AUTHORIZATION: bearer {GITHUB_TOKEN}"
        _run(
            ["git", "-c", f"http.https://github.com/.extraheader={auth_header}",
             "push", "origin", "HEAD"],
            cwd=PROJECT_DIR,
            redact=[GITHUB_TOKEN],
        )
        print("✅ تم رفع التقارير على GitHub بنجاح.")
